# Minimal dataset onboarding example

In [ ]:
import scanpy as sc
import os
from pathlib import Path
import pickle

from preprocessing_utils import (
    filter_cells_by_pert_effect,
    define_splits_singles
)


## 1. Download and load dataset

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE264667

HepG2 cell line

`wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE264nnn/GSE264667/suppl/GSE264667%5Fhepg2%5Fraw%5Fsinglecell%5F01.h5ad`

In [ ]:
base_dir = "."
path_to_dataset = os.path.join(base_dir, "GSE264667_hepg2_raw_singlecell_01.h5ad")

adata = sc.read_h5ad(path_to_dataset)

## 2. Format .obs

The obs dataframe needs to have a few columns to properly encode samples
for interpretation by TxPert

- "condition" a "+" separated list of the gene symbol of perturbation target, and "ctrl"
- "batch" the column with the experimental batch with which cells can be matched to the closest control
- "control" a boolean column indicating whether the observation is of a control or not
- "cell_line" a column indicate the cell line (or type, or more generally the biological context)
- "condition_name" a column with {cell_line}_{condition}_1+1


In [ ]:
## rename columns
# conditions define the perturbation & condtions
# batch is the experimental batch (gemgroup, plate, etc.)
adata.obs.rename(columns={"gene": "condition", "gem_group" : "batch"}, inplace=True)

## create boolean control column
# non-targeting is often the identifier for controls
control_identifier = "non-targeting"
adata.obs["control"] = (adata.obs["condition"] == control_identifier).astype(int)

# append base state to condition name
adata.obs["condition"] = [c + '+ctrl' for c in adata.obs["condition"]]

# rename control
mapper = {k:k for k in adata.obs["condition"].unique()}
mapper[f"{control_identifier}+ctrl"] = "ctrl"
adata.obs["condition"] = adata.obs["condition"].map(mapper)

## make sure that cell type columns exists
cell_line_name = "hepg2"
adata.obs["cell_line"] = cell_line_name # add if missing

# add condition_name ()
adata.obs["condition_name"] = [f"{cell_line_name}_{c}_1+1" for c in adata.obs["condition"]]


#### Confirm all required columns are present

In [ ]:
assert "condition" in adata.obs.columns, "condition column is required (gene symbol of target gene)"
assert "batch" in adata.obs.columns, "batch column is required"
assert "control" in adata.obs.columns, "control column is required"
assert "cell_line" in adata.obs.columns, "cell_line column is required"

## 3. Filter perturbations

In [ ]:
_, adata = filter_cells_by_pert_effect(adata)

## 4. Normalize values 

In [ ]:
sc.pp.normalize_total(adata, target_sum=4000)
sc.pp.log1p(adata)

## 5. Highly variable gene selection

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000, subset=True)

## 6. Define splits

TxPert code will utilize unique values in the 'condition' column of the obs
to both split the data for training, validation, and test; and to breakdown
metrics into subgroups (e.g. for differentiating scores on double perturbations
from single, or scores on where one has seen neither, either, or both of the components
of a double). For both consistency between runs, and adaptability for different datasets, we'll generate and save the splits in advance. 

In [ ]:
unique_conditions = sorted(list(adata.obs['condition'].unique()))
train_test_val, subgroups = define_splits_singles(unique_conditions)
# subgroups is a placeholder here, as all conditions are unseen singles
# it may be customized, as e.g. taking the format of GEARS for Norman data 

#

## 7. Export to expected location

Dataset can now be used for inference or training, remaining formatting will be performed by TxPert.

In [ ]:
CACHE_DIR = Path(os.getcwd()).parent / "cache"
NEW_DATA_DIR = CACHE_DIR / "HepG2_single_cell_line"
SPLITS_DIR = NEW_DATA_DIR / "splits"

os.makedirs(NEW_DATA_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

save_path = NEW_DATA_DIR / "de_adata_test.h5ad"
adata.write(save_path)

with open(SPLITS_DIR / "train_test_split.pkl", "wb") as f:
    pickle.dump(train_test_val, f)

with open(SPLITS_DIR / "subgroup.pkl", "wb") as f:
    pickle.dump(subgroups, f)


To utilize the new dataset, set the following datamodule parameters, either in the config file or on the command line

```
python main.py --config-name=config-gat \
     datamodule.task_type=HepG2_single_cell_line \
     datamodule.train_cell_types=hepg2 \
     datamodule.test_cell_type=hepg2
```

Additionally, if the new cell type is not within the bounds of the original data
which TxPert was developed and tested on, 
you will need to set `suppress_cell_type_validation=true`, or update `constants.py` 
so the code can expect and verify additional cell types/lines. 